In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Dados Históricos - Ibovespa2.csv", parse_dates=[0], index_col=0, decimal=',', thousands='.') # Alteração: ajustando a colunda de data e colocando como index 
df.index = pd.to_datetime(df.index, format = "%d.%m.%Y")
df = df.sort_index()

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 500 entries, 2024-03-01 to 2026-03-02
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Último    500 non-null    int64 
 1   Abertura  500 non-null    int64 
 2   Máxima    500 non-null    int64 
 3   Mínima    500 non-null    int64 
 4   Vol.      500 non-null    object
 5   Var%      500 non-null    object
dtypes: int64(4), object(2)
memory usage: 27.3+ KB


In [4]:
df.dropna()

,Último,Abertura,Máxima,Mínima,Vol.,Var%
Data,,,,,,
2024-03-01,129180,129026,129716,128717,"9,77M","0,12%"
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%"
2024-03-05,128098,128336,128989,127823,"9,69M","-0,19%"
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%"
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%"
...,...,...,...,...,...,...
2026-02-24,191490,188854,191781,188854,"11,04B","1,40%"
2026-02-25,191247,191491,192624,190419,"9,13B","-0,13%"
2026-02-26,191005,191248,191978,188977,"8,80B","-0,13%"


## Definição do Target

O target é definido com base na variação diária do preço de fechamento.

In [5]:
df["y"] = np.sign(df["Último"].diff())
df["y"] = (df["y"] > 0).astype(int)

df = df.dropna()

Este procedimento transforma a variação diária em uma variável binária:

- 1 → dia de alta

- 0 → dia de queda

A primeira linha da série é removida devido à ausência de um valor anterior para cálculo da diferença.

In [6]:
df[["Último","y"]].head(10)

,Último,y
Data,,
2024-03-01,129180,0
2024-03-04,128341,0
2024-03-05,128098,0
2024-03-06,128890,1
2024-03-07,128340,0
2024-03-08,127071,0
2024-03-11,126124,0
2024-03-12,127668,1
2024-03-13,128006,1


In [7]:
df["y"].value_counts(normalize=True)

y
1    0.524
0    0.476
Name: proportion, dtype: float64

## Engenharia de Atributos (Feature Engineering)
A engenharia de atributos tem como objetivo transformar os dados brutos de preço em variáveis que capturem padrões relevantes do comportamento do mercado.

Todas as features são defasadas em um período (shift(1)) para evitar data leakage, garantindo que apenas informações disponíveis até o dia anterior sejam utilizadas na previsão.

### Retornos Recentes

In [8]:
df["retorno_1"] = df["Último"].pct_change().shift(1)
df["retorno_3"] = df["Último"].pct_change(3).shift(1)

**Interpretação**

Retornos representam a variação percentual do preço.

Essas variáveis capturam o comportamento recente do mercado.

**retorno_1**

Representa o retorno percentual do último pregão.

In [9]:
df["vol_5"] = df["Último"].pct_change().rolling(5).std().shift(1)
df["vol_10"] = df["Último"].pct_change().rolling(10).std().shift(1)

In [10]:
df

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10
Data,,,,,,,,,,,
2024-03-01,129180,129026,129716,128717,"9,77M","0,12%",0,NaN,NaN,NaN,NaN
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%",0,NaN,NaN,NaN,NaN
2024-03-05,128098,128336,128989,127823,"9,69M","-0,19%",0,-0.006495,NaN,NaN,NaN
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%",1,-0.001893,NaN,NaN,NaN
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%",0,0.006183,-0.002245,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24,191490,188854,191781,188854,"11,04B","1,40%",1,-0.008823,0.015251,0.010251,0.011319
2026-02-25,191247,191491,192624,190419,"9,13B","-0,13%",0,0.013963,0.015679,0.010366,0.011780
2026-02-26,191005,191248,191978,188977,"8,80B","-0,13%",0,-0.001269,0.003742,0.010164,0.010896


### Médias Móveis

In [11]:
# Calculando a diferença da tendencias
df["ma_5"] = df["Último"].rolling(5).mean().shift(1)

df["ma_10"] = df["Último"].rolling(10).mean().shift(1)
df["ma_diff"] = df["ma_5"] - df["ma_10"]

**Interpretação**

Médias móveis para identificar tendências.

- MA5 → tendência de curtíssimo prazo

- MA10 → tendência de curto prazo

In [12]:
df

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff
Data,,,,,,,,,,,,,,
2024-03-01,129180,129026,129716,128717,"9,77M","0,12%",0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-04,128341,129176,129307,128278,"7,97M","-0,65%",0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-05,128098,128336,128989,127823,"9,69M","-0,19%",0,-0.006495,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-06,128890,128099,129323,128099,"11,06M","0,62%",1,-0.001893,NaN,NaN,NaN,NaN,NaN,NaN
2024-03-07,128340,128890,129188,128033,"7,35M","-0,43%",0,0.006183,-0.002245,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24,191490,188854,191781,188854,"11,04B","1,40%",1,-0.008823,0.015251,0.010251,0.011319,188080.2,187298.6,781.6
2026-02-25,191247,191491,192624,190419,"9,13B","-0,13%",0,0.013963,0.015679,0.010366,0.011780,189085.4,188152.6,932.8
2026-02-26,191005,191248,191978,188977,"8,80B","-0,13%",0,-0.001269,0.003742,0.010164,0.010896,190131.6,188653.2,1478.4


### Diferença entre Médias Móveis

In [13]:
# calculando a diferença de range
df["range_diff"] = (df["Máxima"] - df["Mínima"]).shift(1)

In [56]:
# 1. Distâncias Relativas (Substituindo ma_5 e ma_10 brutos)
df["dist_ma_5"] = ((df["Último"] - df["Último"].rolling(5).mean()) / df["Último"].rolling(5).mean()).shift(1)
df["dist_ma_10"] = ((df["Último"] - df["Último"].rolling(10).mean()) / df["Último"].rolling(10).mean()).shift(1)

# 2. Pressão de Compra/Venda (Onde fechou dentro do candle anterior)
df["pressao_fechamento"] = ((df["Último"] - df["Mínima"]) / (df["Máxima"] - df["Mínima"] + 1e-9)).shift(1)

Essa variável mede a distância entre duas médias móveis.

**Interpretação**

| Valor | Significado |
|------|-------------|
| Positivo | Tendência de alta |
| Negativo | Tendência de baixa |



In [57]:
df["momentum_5"] = (df["Último"] - df["Último"].shift(5)).shift(1)

In [58]:
df.dropna(inplace=True)

As primeiras linhas da série é removida devido à ausência de um valor anterior para cálculo da diferença.

## Separando teste e treino

Diferentemente de problemas tradicionais de machine learning, séries temporais não devem utilizar divisão aleatória.

O conjunto de teste deve conter os dados mais recentes.

Conforme especificado no desafio, os últimos 30 dias foram reservados para teste.

In [59]:
# criando base de teste com as ultimas 30 linhas
test = df.tail(30)
test

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff,range_diff,momentum_5,dist_ma_5,dist_ma_10,pressao_fechamento
Data,,,,,,,,,,,,,,,,,,,
2026-01-16,164800,165557,165872,164100,"11,49B","-0,46%",0,0.002555,0.014821,0.009980,0.008954,163841.4,163019.1,822.3,1237.0,2632.0,0.010538,0.015636,0.594179
2026-01-19,164849,164800,165155,164265,"4,80B","0,03%",1,-0.004639,0.017454,0.010603,0.009039,164127.4,163445.2,682.2,1772.0,1430.0,0.004098,0.008289,0.395034
2026-01-20,166277,164846,166468,163575,"8,37B","0,87%",1,0.000297,-0.001798,0.010506,0.008837,164467.2,163743.1,724.1,890.0,1699.0,0.002321,0.006754,0.656180
2026-01-21,171817,166278,171969,166278,"14,07B","3,33%",1,0.008662,0.004282,0.009312,0.008586,165328.0,164004.4,1323.6,2893.0,4304.0,0.005740,0.013857,0.933979
2026-01-22,175589,171817,177742,171817,"14,02B","2,20%",1,0.033318,0.042579,0.014918,0.012181,166662.2,164988.6,1673.6,5691.0,6671.0,0.030930,0.041387,0.973291
2026-01-23,178859,175590,180532,175590,"11,78B","1,86%",1,0.021954,0.065151,0.015640,0.013186,168666.4,166253.9,2412.5,5925.0,10021.0,0.041043,0.056150,0.636624
2026-01-26,178721,178859,179543,177694,"9,57B","-0,08%",0,0.018623,0.075669,0.012659,0.013486,171478.2,167802.8,3675.4,4942.0,14059.0,0.043042,0.065888,0.661473
2026-01-27,181919,178852,183360,178852,"11,34B","1,79%",1,-0.000772,0.040182,0.013007,0.013438,174252.6,169359.9,4892.7,1849.0,13872.0,0.025643,0.055273,0.555435
2026-01-28,184691,181921,185065,181921,"11,86B","1,52%",1,0.017894,0.036050,0.012276,0.012321,177381.0,171354.5,6026.5,4508.0,15642.0,0.025583,0.061653,0.680346


In [60]:
# montando base de treino sem as ultimas 30 linhas
train = df.iloc[:-30]
train

,Último,Abertura,Máxima,Mínima,Vol.,Var%,y,retorno_1,retorno_3,vol_5,vol_10,ma_5,ma_10,ma_diff,range_diff,momentum_5,dist_ma_5,dist_ma_10,pressao_fechamento
Data,,,,,,,,,,,,,,,,,,,
2024-04-02,127549,126990,127654,126669,"9,07M","0,44%",1,-0.008712,0.001001,0.005699,0.007019,127316.2,127537.5,-221.3,1887.0,-37.0,-0.002562,-0.004293,0.115527
2024-04-03,127318,127546,127694,126181,"11,03M","-0,18%",0,0.004402,-0.001112,0.005997,0.007134,127439.8,127597.0,-157.2,985.0,618.0,0.000857,-0.000376,0.893401
2024-04-04,127428,127313,129627,127178,"13,24M","0,09%",1,-0.001811,-0.006151,0.006104,0.007016,127530.8,127575.9,-45.1,1513.0,455.0,-0.001669,-0.002022,0.751487
2024-04-05,126795,127422,127432,126394,"9,10M","-0,50%",0,0.000864,0.003449,0.005221,0.005479,127478.2,127406.2,72.0,2449.0,-263.0,-0.000394,0.000171,0.102082
2024-04-08,128857,126796,129178,126796,"8,13M","1,63%",1,-0.004968,-0.005911,0.005076,0.005216,127216.0,127269.8,-53.8,1038.0,-1311.0,-0.003309,-0.003731,0.386320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-09,163370,162938,164263,162638,"8,30B","0,27%",1,0.005933,0.006586,0.008955,0.007570,162196.8,161209.4,987.4,1188.0,1811.0,0.004557,0.010710,1.000000
2026-01-12,163150,163370,163493,162277,"6,90B","-0,13%",0,0.002664,-0.001796,0.008338,0.007376,162763.0,161732.2,1030.8,1625.0,2831.0,0.003729,0.010127,0.450462
2026-01-13,161973,163146,163146,161765,"9,67B","-0,72%",0,-0.001347,0.007254,0.008072,0.006295,163019.0,162001.6,1017.4,1216.0,1280.0,0.000804,0.007089,0.717928


## Definição das Variáveis de Entrada

In [61]:




features = ["retorno_1", "retorno_3", "ma_10", "ma_5", "ma_diff","momentum_5", "range_diff", "dist_ma_5", "dist_ma_10", "pressao_fechamento"]

# definindo x e y de treino
X_train = train[features]
y_train = train["y"]

# definindo x e y de teste
X_test = test[features]
y_test = test["y"]

## Modelos e Metricas Avaliadas 

In [62]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [63]:
from sklearn.linear_model import LogisticRegression

model_log = LogisticRegression(max_iter=1000)
model_log.fit(X_train, y_train)

pred_log = model_log.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_log))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_log))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_log))

Accuracy: 0.5666666666666667

Matriz de confusão:  [[ 0 13]
 [ 0 17]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.57      1.00      0.72        17

    accuracy                           0.57        30
   macro avg       0.28      0.50      0.36        30
weighted avg       0.32      0.57      0.41        30



/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [64]:
from sklearn.tree import DecisionTreeClassifier

model_tree = DecisionTreeClassifier(random_state=42)
model_tree.fit(X_train, y_train)

pred_tree = model_tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_tree))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_tree))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_tree))

Accuracy: 0.6

Matriz de confusão:  [[12  1]
 [11  6]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.52      0.92      0.67        13
           1       0.86      0.35      0.50        17

    accuracy                           0.60        30
   macro avg       0.69      0.64      0.58        30
weighted avg       0.71      0.60      0.57        30



In [65]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred_rf))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_rf))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_rf))

Accuracy: 0.6666666666666666

Matriz de confusão:  [[ 8  5]
 [ 5 12]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.62      0.62      0.62        13
           1       0.71      0.71      0.71        17

    accuracy                           0.67        30
   macro avg       0.66      0.66      0.66        30
weighted avg       0.67      0.67      0.67        30



In [66]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [67]:
from sklearn.svm import SVC

model_svm = SVC()
model_svm.fit(X_train_scaled, y_train)

pred_svm = model_svm.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_svm))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_svm))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_svm))

Accuracy: 0.5666666666666667

Matriz de confusão:  [[ 0 13]
 [ 0 17]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        13
           1       0.57      1.00      0.72        17

    accuracy                           0.57        30
   macro avg       0.28      0.50      0.36        30
weighted avg       0.32      0.57      0.41        30



/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/root/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [68]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [69]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled, y_train)

pred_knn = model_knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, pred_knn))
print('\nMatriz de confusão: ', confusion_matrix(y_test, pred_knn))
print("\nRelatório de Classificação:\n", classification_report(y_test, pred_knn))

Accuracy: 0.6333333333333333

Matriz de confusão:  [[ 8  5]
 [ 6 11]]

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.57      0.62      0.59        13
           1       0.69      0.65      0.67        17

    accuracy                           0.63        30
   macro avg       0.63      0.63      0.63        30
weighted avg       0.64      0.63      0.63        30



In [70]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [71]:
resultados = {
    "Logistic Regression": accuracy_score(y_test, pred_log),
    "Decision Tree": accuracy_score(y_test, pred_tree),
    "Random Forest": accuracy_score(y_test, pred_rf),
    "KNN": accuracy_score(y_test, pred_knn),
    "SVM": accuracy_score(y_test, pred_svm)
}

pd.Series(resultados).sort_values(ascending=False)

Random Forest          0.666667
KNN                    0.633333
Decision Tree          0.600000
Logistic Regression    0.566667
SVM                    0.566667
dtype: float64

In [72]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix